# napari

napari is an interactive, multi-dimensional open-source image viewer based on Python

[GitHub](https://github.com/napari/napari) - [Docs](https://napari.org/stable/index.html)

---

## napari GUI (15 min)

<div style="border-left: 4px solid #4CAF50; padding-left: 1em;">

**TASK**: Explore napari GUI capabilities - interactive image inspection and annotation

- Activate the *dl_tools* environment made with the instructions in README dile
- Start the napari GUI - run: `napari`
- Explore the GUI together

</div>

NOTE: You can discover napari plugins at: [napari-hub](https://napari-hub.org/)

---

## empanada-napari (30 min)

**Empanada** is a deep-learning based tool for organelle segmentation in 2D and 3D electron microscopy images.

It is integrated in napari as a plugin.

[GitHub](https://github.com/volume-em/empanada-napari) - [Docs](https://empanada.readthedocs.io/en/latest/index.html)

<br>

Example data sources: https://www.ebi.ac.uk/empiar/EMPIAR-10982/



<div style="border-left: 4px solid #4CAF50; padding-left: 1em;">

**TASK**: Explore deep-learning segmentation in napari using *empanada-napari*

*2D*
- Open the image **demo_image** from the folder `empanada_example`
- Start 2D inference from empanada plugin menu
- Run a pre-trained empanada model: `MitoNet_v1`
- Test different parameters in the plugin menu
- Try other pre-trained models (NucleoNet or DropNet)

*3D*
- Remove all existing layers in napari
- Open the image **lucchi_pp_3d.tif** from the folder `napari_example`
- Start 3D inference from empanada plugin menu
- Run a pre-trained empanada model: `MitoNet_v1_mini`


</div>

NOTES:

- You can re-train (fine-tune) the model on your own dataset, or train a new panoptic segmentation model - [manual](https://empanada.readthedocs.io/en/latest/tutorials/train_panop.html#train-panoptic-model).
- The panoptic segmentation approach combines semantic and instance segmentation. 
- In **Empanada**, distinguishing classes and instances within the same mask is handled using label offsets (divisors) - e.g. [100, 200, 300..] define the semantic class and [101, 102, 201…] represent individual instances within given class.

---

## napari from code (15 min)

**Jupyter-napari bridge**

The core concept is simple:
- We import napari and create a viewer object in our notebook
- The viewer object provides programmatic access to the napari window (GUI)
- We can send data from Python to the GUI and access GUI layers and annotations directly from the notebook

<div style="border-left: 4px solid #4CAF50; padding-left: 1em;">

**TASK**: Using napari from Jupter Notebook

- Import the required libraries (`napari`, `numpy`, `matplotlib`, `skimage.io`)
- Load the image `slice` from the folder `empanada_example`
- Initialize a napari viewer
- Add the image to the viewer as an *image* layer
- Perform simple thresholding to create a binary mask
- Add the mask to napari as a *labels* layer
- Modify the mask manually using the napari GUI tools
- Display the modified mask in the notebook using `matplotlib.pyplot`

</div>

In [ ]:
# Your code here




<details>
<summary><b>💡 Example solution</b></summary>

```python
# Import the napari package
import napari

# Import other libraries
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread

# Create a new viewer instance
viewer = napari.Viewer() 

# Load image
image = imread(r'../data/cellpose_example/Stardist_1.tif') # read image specified by path
image_layer = viewer.add_image(image)

# Segment image
threshold = 3000
mask = np.array(image > threshold, dtype='uint8')

# Load mask to napari
mask_layer = viewer.add_labels(mask, name='threshold')


for layer in viewer.layers:
    print(layer) # prints the currently existing layers


# Modify mask in napari
# ...


# Display modified image in notebook
plt.imshow(mask_layer.data, cmap='nipy_spectral')

# Create new label layer in napari and 
# ...

# Read napari layer to notebook
mask_new = viewer.layers['Labels'].data
plt.imshow(mask_new, cmap='nipy_spectral')

```

</details>

---

### **Bonus** - ✨🧙 Custom widgets 🧙✨


This example demonstrates how to create an interactive thresholding tool in `napari` using `magicgui`.

The `@magicgui` decorator automatically turns a Python function into a widget — in this case a slider widget - that lets you adjust the threshold value dynamically.

In [ ]:
import napari
from skimage import measure, io

# Load image
image_path = r'../data/cellpose_example/Stardist_1.tif'
image = io.imread(image_path)

# Initialize napari & load image layer
viewer = napari.Viewer()
viewer.add_image(image, name='Raw Image')

In [ ]:
from magicgui import magicgui


# Minimal threshold slider
@magicgui(threshold={'widget_type': 'FloatSlider', 'min': 0,'max': image.max(),'step': 1})
def apply_threshold(threshold):
    mask = (image > threshold)
    labels = measure.label(mask)

    # Update or create mask layer
    if 'Thr' in viewer.layers:
        viewer.layers['Thr'].data = mask
    else:
        viewer.add_labels(mask, name='Thr')
    
    # Update or create label layer
    if 'Thr_Labels' in viewer.layers:
        viewer.layers['Thr_Labels'].data = labels
    else:
        viewer.add_labels(labels, name='Thr_Labels')

# Add widget to napari window
viewer.window.add_dock_widget(apply_threshold)